# RSM Gridder — Offline and Live

Binning scattered reciprocal-space points into a regular HKL volume.

- **Offline** (`rsm_gridder`) — merge saved scans; bounds, memory budgeting, masks. What the RSM Volume Builder runs.
- **Live** (`rsm_live_grid`) — accumulate frame by frame during a scan. What the HKL 3D viewer's *Gridded volume* mode shows.

Cells needing a real scan report and continue.

**Setup.** `rsm_gridder` is functions, not a class. Set `FILES` and the voxel counts.

In [ ]:
import os

import numpy as np

from dashpva.utils import rsm_gridder as gridder
from dashpva.utils.rsm_gridder import GridBounds
from dashpva.utils import HDF5Loader, MaskManager, RSMConverter

FILES = [os.path.expanduser(p) for p in
         ["~/DashPVA/outputs/file1.h5", "~/DashPVA/outputs/file2.h5"]]   # <- one or more scans to merge
NX, NY, NZ = 128, 128, 128          # voxels along H, K, L

print("batch default:", gridder.DEFAULT_BATCH_BYTES // 2**20, "MiB")

## 1. Budget first

Grid + coverage + working batch. Easy to ask for more RAM than you have.

**Estimate.** Note doubling each axis is 8× the grid, not 2×.

In [ ]:
est = gridder.estimate_grid_memory(NX, NY, NZ, detector_shapes=[(516, 516)])
mib = lambda b: f"{b / 2**20:,.0f} MiB"
print("peak  :", mib(est.peak_bytes))
print("  grid:", mib(est.grid_bytes))
print("  batch:", mib(est.batch_bytes))
print("  output:", mib(est.output_bytes))

# Doubling each axis is 8x the grid, not 2x
big = gridder.estimate_grid_memory(NX*2, NY*2, NZ*2, detector_shapes=[(516, 516)])
print("\n2x per axis ->", mib(big.peak_bytes))

**Plot the budget.** Cubic growth is hard to feel from two numbers.

In [ ]:
import matplotlib.pyplot as plt

# Why the 8x matters: memory is cubic in voxels-per-axis. Log y, or the small
# grids vanish. Two series, so a legend; one axis, never two scales.
ns = [32, 64, 96, 128, 160, 192, 224, 256]
ests = [gridder.estimate_grid_memory(n, n, n, detector_shapes=[(516, 516)]) for n in ns]
peak = [e.peak_bytes / 2**20 for e in ests]
grid = [e.grid_bytes / 2**20 for e in ests]

fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.plot(ns, peak, marker="o", linewidth=2, color="#0072B2", label="peak")
ax.plot(ns, grid, marker="o", linewidth=2, color="#D55E00", label="grid only")
ax.set_yscale("log")
ax.set_xlabel("voxels per axis (N, for an N x N x N grid)")
ax.set_ylabel("MiB (log)")
ax.axvline(NX, color="0.6", linewidth=1, linestyle="--")
ax.annotate(f"current N = {NX}", xy=(NX, max(peak)), xytext=(4, -8),
            textcoords="offset points", fontsize=9, color="0.35")
ax.grid(axis="y", alpha=0.25)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.legend(frameon=False)
plt.show()

**Estimate from files.** Cheap — reads only detector shapes. `ensure_memory_available` raises before a long build, not an hour in.

In [ ]:
try:
    est = gridder.estimate_files_memory(FILES, NX, NY, NZ)
    print("estimated peak:", mib(est.peak_bytes))
    gridder.ensure_memory_available(est, max_fraction=0.7)
    print("within budget")
except MemoryError as e:
    print("Would not fit:", e)
except Exception as e:
    print("Cannot estimate yet:", type(e).__name__, e)

## 2. Bounds

The grid needs an HKL box before binning starts.

**Discover.** A full pass computing Q without binning. Also returns energy/UB/frame count for the consistency check.

In [ ]:
try:
    conv = RSMConverter()
    seen = []
    bounds, info = gridder.compute_file_bounds(
        FILES[0], conv, progress_cb=lambda d, t: seen.append((d, t)))
    print("bounds:", bounds)
    print("frames:", info.num_frames, "detector:", info.detector_shape)
    print("energy:", info.energy_eV, "eV")
except Exception as e:
    print("Bounds pass unavailable:", type(e).__name__, e)

**Or fix them.** Skips the discovery pass and makes two builds comparable. Points outside are dropped.

In [ ]:
fixed = GridBounds(xmin=-1.0, xmax=1.0,
                   ymin=-1.0, ymax=1.0,
                   zmin=0.0,  zmax=2.0)
print("fixed box:", fixed)
print("voxel size:", (fixed.xmax - fixed.xmin) / NX,
                     (fixed.ymax - fixed.ymin) / NY,
                     (fixed.zmax - fixed.zmin) / NZ)

## 3. Build

**Build.** `warn` catches non-fatal issues (mismatched energy, etc.) so they don't vanish into a log.

In [ ]:
def on_warn(msg):
    print("  warning:", msg)

try:
    result = gridder.build_volume(
        FILES, nx=NX, ny=NY, nz=NZ,
        use_mask=False,
        progress_cb=lambda d, t: None,
        warn=on_warn,
    )
    print("volume    :", result.volume.shape, result.volume.dtype)
    print("binned    :", f"{result.num_points_binned:,}")
    print("excluded  :", f"{result.num_points_excluded_nonfinite:,} non-finite")
    print("axes      :", result.xaxis.shape, result.yaxis.shape, result.zaxis.shape)
except Exception as e:
    print("Build unavailable:", type(e).__name__, e)

**NaN, not zero.** Unmeasured ≠ measured zero. Use `finite_intensity_range()` for colour limits or NaNs flatten the scale.

In [ ]:
try:
    vol = result.volume
    filled = np.count_nonzero(np.isfinite(vol))
    print(f"voxels filled: {filled:,} / {vol.size:,} ({100*filled/vol.size:.1f}%)")
    print("intensity range (finite only):", gridder.finite_intensity_range(vol))
    print("coverage array:", None if result.coverage is None else result.coverage.shape)
except NameError:
    print("No volume built in this session")

**Look at it.** A projection plus the intensity histogram — the histogram is
where you notice a monitor problem or a mask you forgot to apply.

In [ ]:
import matplotlib.pyplot as plt

# Guarded: only runs if the build above produced a volume.
if "result" in dir() and getattr(result, "volume", None) is not None:
    vol = result.volume
    finite = vol[np.isfinite(vol)]

    fig, (ax_img, ax_hist) = plt.subplots(
        1, 2, figsize=(11, 3.8), constrained_layout=True,
        gridspec_kw={"width_ratios": [1, 1.2]})

    mid = vol.shape[2] // 2
    im = ax_img.imshow(np.nanmax(vol, axis=2).T, origin="lower",
                       aspect="auto", cmap="magma")
    ax_img.set_xlabel("H index"); ax_img.set_ylabel("K index")
    ax_img.set_title("max projection along L", fontsize=10)
    fig.colorbar(im, ax=ax_img, shrink=0.85, label="intensity")

    # Detector counts span decades -- a linear axis would show one spike.
    ax_hist.hist(finite, bins=80, log=True, color="#0072B2")
    ax_hist.set_xlabel("intensity"); ax_hist.set_ylabel("voxels (log)")
    ax_hist.set_title(
        f"{finite.size:,} finite of {vol.size:,} voxels "
        f"({100 * finite.size / vol.size:.1f}% filled)", fontsize=10)
    for side in ("top", "right"):
        ax_hist.spines[side].set_visible(False)
    plt.show()
else:
    print("No volume built yet -- point FILES at a real scan and re-run the build cell.")

**3D.** Nested isosurfaces of the same volume — the projections show where
intensity is, this shows its shape. Needs `pyvista[jupyter]` (the standalone extra).

In [ ]:
import numpy as np
import pyvista  # not `as pv` -- `pv` is the live preview payload above

# Prefer the offline build; fall back to the live accumulator.
_vol = None
if "result" in dir() and getattr(result, "volume", None) is not None:
    _vol, _origin, _spacing = result.volume, None, None
elif "pv" in dir() and getattr(pv, "mean", None) is not None:
    _vol, _origin, _spacing = pv.mean, pv.origin, pv.spacing

if _vol is None:
    print("No volume in scope -- run the build cell or the live-grid cell first.")
else:
    finite = _vol[np.isfinite(_vol)]
    grid = pyvista.ImageData(dimensions=np.array(_vol.shape) + 1)
    if _origin is not None:
        grid.origin, grid.spacing = _origin, _spacing
    grid.cell_data["intensity"] = np.nan_to_num(
        _vol, nan=float(np.nanmin(finite))).flatten(order="F")

    # Contour the POINT data, so take the levels from the point data too:
    # cell->point averaging lowers the peaks, and a level computed on the cell
    # array can land above the smoothed maximum and contour to nothing.
    points = grid.cell_data_to_point_data()
    scal = points["intensity"]
    levels = [float(v) for v in np.percentile(scal, [92.0, 97.0, 99.5])]
    # Keep every level strictly inside the range, else that shell is empty.
    lo, hi = float(scal.min()), float(scal.max())
    levels = sorted({min(max(v, lo + 1e-6 * (hi - lo)), hi - 1e-6 * (hi - lo))
                     for v in levels})

    mesh = points.contour(levels, scalars="intensity")
    print(f"isosurfaces at {[f'{v:.3g}' for v in levels]} -> "
          f"{mesh.n_points:,} points, {mesh.n_faces:,} faces")

    if mesh.n_points == 0:
        print("Empty contour -- the volume is too flat at these levels. "
              "Lower the percentiles, or check the intensity range.")
    else:
        plotter = pyvista.Plotter(window_size=(720, 560))
        plotter.add_mesh(mesh, scalars="intensity", cmap="magma",
                         opacity=0.45, smooth_shading=True,
                         scalar_bar_args={"title": "intensity"})
        plotter.show_grid(xtitle="H", ytitle="K", ztitle="L")
        plotter.camera_position = "iso"
        plotter.show()

## 4. Masking and normalisation

Both act per pixel *before* binning.

**Mask.** Masked pixels never enter the grid — better than zeroing after, which drags voxel means down.

In [ ]:
import tempfile

# Notebook-local mask dir so nothing here touches your real masks/active_mask.npy
MASKS_DIR = tempfile.mkdtemp(prefix="dashpva_nb_masks_")
mask_mgr = MaskManager(masks_dir=MASKS_DIR)
# mask_mgr.load_mask("masks/detector_hot.npy", detector_shape=(516, 516))

# result = gridder.build_volume(
#     FILES, nx=NX, ny=NY, nz=NZ,
#     use_mask=True, mask_manager=mask_mgr, mask_transposed=False,
# )
# print("excluded by mask:", f"{result.num_points_excluded_by_mask:,}")
print("mask currently loaded:", mask_mgr.mask is not None)

**Monitor.** Beam drifts over a long scan. `list_monitor_candidates` shows what a file actually has.

In [ ]:
try:
    cands = gridder.list_monitor_candidates(FILES[0])
    print("monitor candidates:", cands or "(none recorded)")
except Exception as e:
    print("Cannot list monitors:", type(e).__name__, e)

# result = gridder.build_volume(FILES, NX, NY, NZ, monitor_dataset="Filter_Trans")

## 5. Merging scans

Several filenames bin into one grid; bounds cover the union.

**Check first.** Different energy or UB means different reciprocal space. This *warns*, it does not block — read them.

In [ ]:
infos = []
for name in FILES:
    try:
        _, info = gridder.compute_file_bounds(name, RSMConverter())
        infos.append(info)
    except Exception as e:
        print(f"  {name}: {type(e).__name__}")

if infos:
    warnings = gridder.validate_consistency(infos, energy_rtol=1e-4, ub_atol=1e-4)
    print("consistency warnings:", warnings or "none — files agree")

## 6. Save

Axes, bounds and provenance travel with the array so the Workbench can reopen it.

In [ ]:
# meta = gridder.volume_result_to_metadata(result, extra={"note": "merged 3 scans"})
# ok = HDF5Loader().save_vol_to_h5("merged_volume.h5", result.volume,
#                                  metadata=meta, coverage=result.coverage)
# print("saved:", ok)

## 7. The live grid

Gridder3D latches its range on first use, so the box is fixed **before** the first frame — rebinning mid-scan would change what a voxel means. Frames outside are counted, not accommodated.

That's why the grid dock makes you set bounds before Start.

**Accumulate.** `add_frame` bins one frame; `preview` returns a downsampled payload small enough to publish each update.

In [ ]:
from dashpva.utils.rsm_live_grid import GridBoundsSpec, LiveVolumeAccumulator

spec = GridBoundsSpec(hmin=-1.0, hmax=1.0, kmin=-1.0, kmax=1.0,
                      lmin=0.0, lmax=2.0, nx=48, ny=48, nz=48)
acc = LiveVolumeAccumulator(spec)

# Stand-in for a scan: three Bragg-like peaks, a frame at a time
rng = np.random.default_rng(0)
PEAKS = [(-0.45, -0.35, 0.70), (0.30, 0.10, 1.15), (0.05, 0.55, 1.60)]

for _ in range(40):
    q, w = [], []
    for cx, cy, cz in PEAKS:
        pts = rng.normal([cx, cy, cz], 0.07, size=(1500, 3))
        q.append(pts)
        w.append(rng.random(1500).astype(np.float32) * 100)
    q.append(rng.uniform([-1, -1, 0], [1, 1, 2], size=(2000, 3)))   # background
    w.append(rng.random(2000).astype(np.float32) * 5)
    q, w = np.vstack(q), np.concatenate(w)
    acc.add_frame(q[:, 0], q[:, 1], q[:, 2], w)

pv = acc.preview()
print("preview shape  :", pv.shape)
print("voxels filled  :", f"{pv.voxels_filled:,}")
print("points binned  :", f"{pv.points_binned:,}")
print("intensity range:", [round(v, 1) for v in pv.intensity_range])

**Look at it.** Three orthogonal max projections of the accumulated grid.

In [ ]:
import matplotlib.pyplot as plt

# Three orthogonal max-intensity projections of the live grid. Magnitude, so a
# single perceptually-uniform ramp -- never a rainbow, which invents structure.
vol = pv.mean
ox, oy, oz = pv.origin
dx, dy, dz = pv.spacing
nx, ny, nz = vol.shape
h = (ox, ox + nx * dx)
k = (oy, oy + ny * dy)
ll = (oz, oz + nz * dz)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)
panels = [
    (np.nanmax(vol, axis=2).T, (*h, *k), "H", "K"),   # down L
    (np.nanmax(vol, axis=1).T, (*h, *ll), "H", "L"),  # down K
    (np.nanmax(vol, axis=0).T, (*k, *ll), "K", "L"),  # down H
]
for ax, (img, extent, xl, yl) in zip(axes, panels):
    im = ax.imshow(img, origin="lower", extent=extent, aspect="auto", cmap="magma")
    ax.set_xlabel(xl); ax.set_ylabel(yl)
    ax.set_title(f"max along {({'H','K','L'} - {xl, yl}).pop()}", fontsize=10)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
fig.colorbar(im, ax=axes, shrink=0.85, label="mean intensity")
fig.suptitle(
    f"Live grid after {pv.frames_accepted} frames "
    f"({pv.voxels_filled:,}/{vol.size:,} voxels filled)", fontsize=11)
plt.show()

**In 3D.** The same live grid volume-rendered. Unmeasured voxels are NaN and stay transparent, so you see the peaks rather than a solid block.

In [ ]:
import pyvista   # `pv` is the preview payload, so pyvista is not aliased

vol = pv.mean
finite = vol[np.isfinite(vol)]

grid = pyvista.ImageData(dimensions=np.array(vol.shape) + 1)
grid.origin, grid.spacing = pv.origin, pv.spacing
grid.cell_data["intensity"] = vol.flatten(order="F")

lut = pyvista.LookupTable(cmap="viridis")
lut.scalar_range = (float(finite.min()), float(finite.max()))
lut.nan_color = (0.0, 0.0, 0.0, 0.0)      # unmeasured -> transparent

plotter = pyvista.Plotter(window_size=(720, 560))
plotter.add_volume(grid, scalars="intensity", cmap=lut, opacity="sigmoid")
plotter.show_grid(xtitle="H", ytitle="K", ztitle="L")
plotter.camera_position = "iso"
plotter.show()

**Out of range.** Not an error, but a large count means the bounds were wrong — hence the counter in the dock.

In [ ]:
probe = np.array([[0.0, 0.0, 1.0],     # inside
                  [5.0, 0.0, 1.0],     # outside in H
                  [0.0, 0.0, 9.0]])    # outside in L
inside = spec.contains_mask(probe[:, 0], probe[:, 1], probe[:, 2])
print("inside the box:", inside)

**Geometry guard.** A session stops if the static geometry changes mid-run.

It's sensitive: a value missing on one frame changes the hash as much as a real edit.

In [ ]:
from dashpva.utils.rsm_grid_session import geometry_fingerprint

base    = [("energy", 11.215), ("ub", (1.0, 0.0, 0.0))]
missing = [("energy", None),   ("ub", (1.0, 0.0, 0.0))]

print("all present:", geometry_fingerprint(base, {}, (516, 516)))
print("one missing:", geometry_fingerprint(missing, {}, (516, 516)))
print("-> a dropped value reads as a geometry change")

**Clear.** Resets the sums, keeps the box, so runs stay comparable.

In [ ]:
acc.clear()
print("after clear, preview range:", acc.preview().intensity_range)

## Offline vs live

| | offline | live |
|---|---|---|
| bounds | discovered or fixed | fixed before frame 1 |
| input | saved files | frames as they arrive |
| merging | many files | one run |
| mask / monitor | both | mask only |
| output | full volume + coverage | downsampled preview |
| used by | RSM Volume Builder | analysis consumer, HKL 3D |

Offline for the best volume; live to see something while the scan runs.